# Algoritmos de optimización - Seminario

**Nombre y apellidos:**  
Francis Arthur Poldark Palomino Marino  
Cesar Jose Ortiz Sabogal

**URL:** `https://github.com/FPalomino-gif/Algoritmos-de-optimizacion-seminario`
**Problema seleccionado:** 3. Combinar cifras y operaciones**

## PROBLEMA 3: COMBINAR CIFRAS Y OPERACIONES  

### Descripción del problema

Disponemos de las nueve cifras del 1 al 9, excluyendo el cero, y de los cuatro signos básicos de las operaciones fundamentales: suma (`+`), resta (`-`), multiplicación (`*`) y división (`/`). Debemos combinarlos alternativamente, sin repetir ninguna cifra ni ningún operador, para obtener una cantidad dada. Un ejemplo para obtener el valor 4 es:

`4 + 2 - 6 / 3 * 1 = 4`

El trabajo debe encontrar todos los resultados enteros posibles, determinar sus valores mínimo y máximo y comprobar si pueden obtenerse todos los enteros comprendidos entre ambos.

Según lo que hemos visto en la asignatura de Algoritmos de Optimización, el problema puede modelarse así:

In [1]:
# ==============================================================================
# MODELO DE OPTIMIZACIÓN Y BÚSQUEDA EXHAUSTIVA
# ==============================================================================
# 1. ENTRADAS:
#    - Cifras disponibles : C = {1, 2, 3, 4, 5, 6, 7, 8, 9}
#    - Operadores         : O = {+, -, *, /}
#
# 2. VARIABLES DE DECISIÓN:
#    - Selección y orden de 5 cifras : (n1, n2, n3, n4, n5)
#    - Orden de los 4 operadores     : (op1, op2, op3, op4)
#
# 3. RESTRICCIONES:
#    - Sin cero (0 ∉ C)
#    - Cifras distintas (n_i != n_j)
#    - Operadores distintos (op_i != op_j)
#    - Alternancia estricta (Cifra - Op - Cifra - Op...)
#    - Resultados pertenecientes a Z (enteros exactos)
#
# 4. SALIDAS:
#    - Conjunto de valores enteros alcanzables
#    - Valor Mínimo y Valor Máximo
#    - Evaluación de continuidad (enteros faltantes en el rango)
# ==============================================================================

Debe analizarse el problema para encontrar todos los valores enteros posibles planteando las siguientes cuestiones:

### 1. ¿Qué valor máximo y mínimo se pueden obtener según las condiciones del problema?

El valor mínimo es *−69, que puede obtenerse con 1+4/2-8*9 = -69. El valor máximo es **77*, que puede obtenerse con 7/1-2+8*9 = 77. Ambas expresiones usan cinco cifras diferentes y los cuatro operadores exactamente una vez.

### 2. ¿Es posible encontrar todos los valores enteros posibles entre dicho mínimo y máximo ?

Sí. La enumeración exhaustiva demuestra que se pueden obtener todos los enteros del intervalo cerrado $[-69,77]$. El intervalo contiene

$$77-(-69)+1=147$$

valores enteros y el algoritmo encuentra exactamente *147 resultados enteros distintos*, por lo que no existe ningún valor ausente. Las secciones posteriores muestran el algoritmo y la comprobación reproducible de estas respuestas.

## DESARROLLO DEL PROBLEMA 3

### 3. (*) ¿Cuántas posibilidades hay sin tener en cuenta las restricciones?  

Se conserva el formato de cinco posiciones para cifras y cuatro para operadores, pero se permite repetir ambos. Por tanto:

$$9^5\cdot4^4=59\,049\cdot256=15\,116\,544.$$

Hay **15.116.544 expresiones**. Esta interpretación de «sin restricciones» mantiene la alternancia cifra-operador; si tampoco se fijara la forma o la longitud de la expresión, el número de posibilidades no estaría acotado.

In [2]:
posibilidades_sin_restricciones = 9**5 * 4**4
posibilidades_sin_restricciones

15116544

### 4. ¿Cuántas posibilidades hay teniendo en cuenta todas las restricciones?

Las cinco cifras distintas y ordenadas se eligen mediante una variación sin repetición:

$$P(9,5)=\frac{9!}{(9-5)!}=15\,120.$$

Los cuatro operadores distintos se pueden ordenar de $4!=24$ maneras. Luego:

$$P(9,5)\cdot4!=15\,120\cdot24=362\,880.$$

Hay **362.880 expresiones válidas**.

In [3]:
import math

posibilidades_con_restricciones = math.perm(9, 5) * math.factorial(4)
posibilidades_con_restricciones

362880

## Modelo para el espacio de soluciones  

### 5. (*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, argumentalo)

- **Tuplas:** representan cada ordenación de cifras y operadores. Son apropiadas porque una combinación no debe modificarse y `itertools.permutations` ya las produce.
- **Diccionario:** relaciona cada resultado entero con una expresión de ejemplo. La consulta y la inserción tienen coste medio $O(1)$.
- **Conjunto:** se usa más adelante para comprobar en coste medio $O(1)$ si la quinta cifra calculada está disponible.

Una lista serviría para guardar resultados, pero exigiría búsquedas lineales para evitar duplicados. El diccionario conserva a la vez la unicidad del resultado y una solución legible.

## Según el modelo para el espacio de soluciones:

### 6. (*) ¿Cuál es la función objetivo?

Sea $x$ una expresión válida, $G(x)$ su valor y $T$ el entero objetivo. Se define:

$$f(x)=|G(x)-T|.$$


### 7. (*) ¿Es un problema de maximización o minimización?

Se busca una expresión que minimice $f(x)$. Existe una solución exacta cuando $f(x)=0$. Por tanto, planteado para alcanzar una cantidad dada, es un **problema de minimización**. La enumeración adicional del menor y del mayor resultado es un análisis del espacio de soluciones, no cambia esa función objetivo.

### 8. Diseñe un algoritmo para resolver el problema por fuerza bruta.

Se parte del algoritmo propuesto: generar todas las permutaciones de cinco cifras, todas las permutaciones de operadores, construir la expresión y evaluarla. El parámetro `objetivo` se fija en una celda para que el notebook pueda ejecutarse sin interacción.

#### Pseudocódigo de fuerza bruta

```text
ALGORITMO FuerzaBruta(cifras, operadores, objetivo)
    valores_encontrados ← diccionario vacío
    solucion ← NULO
    total_expresiones ← 0

    PARA CADA numeros EN Permutaciones(cifras, 5) HACER
        PARA CADA operaciones EN Permutaciones(operadores, 4) HACER
            expresion ← ConstruirExpresion(numeros, operaciones)
            resultado ← Evaluar(expresion)
            total_expresiones ← total_expresiones + 1

            SI resultado es entero ENTONCES
                SI resultado NO ESTÁ EN valores_encontrados ENTONCES
                    valores_encontrados[resultado] ← expresion
                FIN SI

                SI resultado = objetivo Y solucion = NULO ENTONCES
                    solucion ← expresion
                FIN SI
            FIN SI
        FIN PARA
    FIN PARA

    DEVOLVER solucion, valores_encontrados, total_expresiones
FIN ALGORITMO
```

El algoritmo no se detiene al encontrar el objetivo porque también debe reunir todos los resultados enteros para calcular el mínimo, el máximo y los posibles huecos del intervalo.

In [4]:
import itertools

cifras = [1, 2, 3, 4, 5, 6, 7, 8, 9]
operadores = ["+", "-", "*", "/"]
objetivo = 10


def construir_expresion(numeros, operaciones):
    """Construye una expresión alternando cinco cifras y cuatro operadores."""
    expresion = str(numeros[0])
    for operador, numero in zip(operaciones, numeros[1:]):
        expresion += operador + str(numero)
    return expresion


def fuerza_bruta(objetivo):
    """Enumera todo el espacio válido y conserva un ejemplo por entero."""
    valores_encontrados = {}
    solucion_encontrada = None
    total_expresiones = 0

    for numeros in itertools.permutations(cifras, 5):
        for operaciones in itertools.permutations(operadores):
            expresion = construir_expresion(numeros, operaciones)
            resultado = eval(expresion)  
            total_expresiones += 1

            # Se tolera el pequeño error propio de la aritmética en coma flotante.
            if abs(resultado - round(resultado)) < 1e-6:
                resultado_entero = round(resultado)
                valores_encontrados.setdefault(resultado_entero, expresion)
                if resultado_entero == objetivo and solucion_encontrada is None:
                    solucion_encontrada = expresion

    return solucion_encontrada, valores_encontrados, total_expresiones


solucion, valores_encontrados, total_expresiones = fuerza_bruta(objetivo)
print(f"Total de expresiones probadas: {total_expresiones:,}".replace(",", "."))
print(f"Solución para {objetivo}: {solucion} = {objetivo}")

Total de expresiones probadas: 362.880
Solución para 10: 1+2*6-9/3 = 10


El siguiente código responde las dos preguntas centrales del problema: obtiene los extremos y comprueba si falta algún entero entre ambos.

### Valores enteros alcanzables

El siguiente código responde las dos preguntas centrales del problema: obtiene los extremos y comprueba si falta algún entero entre ambos.

In [5]:
valor_minimo = min(valores_encontrados)
valor_maximo = max(valores_encontrados)
faltantes = [
    valor for valor in range(valor_minimo, valor_maximo + 1)
    if valor not in valores_encontrados
]

print("Valor mínimo entero:", valor_minimo)
print("Expresión:", valores_encontrados[valor_minimo])
print("Valor máximo entero:", valor_maximo)
print("Expresión:", valores_encontrados[valor_maximo])
print("Cantidad de valores enteros distintos:", len(valores_encontrados))
print("¿Se alcanzan todos los enteros del intervalo?:", not faltantes)
print("Valores ausentes:", faltantes)
print("\nValores enteros encontrados:")
print(sorted(valores_encontrados))

Valor mínimo entero: -69
Expresión: 1+4/2-8*9
Valor máximo entero: 77
Expresión: 7/1-2+8*9
Cantidad de valores enteros distintos: 147
¿Se alcanzan todos los enteros del intervalo?: True
Valores ausentes: []

Valores enteros encontrados:
[-69, -68, -67, -66, -65, -64, -63, -62, -61, -60, -59, -58, -57, -56, -55, -54, -53, -52, -51, -50, -49, -48, -47, -46, -45, -44, -43, -42, -41, -40, -39, -38, -37, -36, -35, -34, -33, -32, -31, -30, -29, -28, -27, -26, -25, -24, -23, -22, -21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77]


### 9. Calcule la complejidad del algoritmo por fuerza bruta.

Para el caso concreto siempre se ejecutan 362.880 evaluaciones. Sin embargo, llamarlo simplemente $O(1)$ ocultaría cómo crece el algoritmo. Si $n$ es el número de cifras disponibles y se siguen escogiendo cinco, el coste es:

$$\Theta(P(n,5)\cdot4!\cdot5)=\Theta(n^5),$$

porque construir/evaluar una expresión de nueve símbolos tiene longitud constante. El diccionario puede guardar como máximo una entrada por resultado entero encontrado.

En una generalización con $k$ cifras usadas y $m$ operadores disponibles, sin repetición, el coste sería $\Theta(P(n,k)\,P(m,k-1)\,k)$. Solo si se considera exclusivamente esta entrada fija —sin ningún parámetro que pueda crecer— el tiempo puede describirse como constante.

### 10. (*) Diseñe un algoritmo que mejore la complejidad del algoritmo por fuerza bruta. Argumente por qué mejora el algoritmo por fuerza bruta.

Para buscar un objetivo no es necesario probar las nueve posibilidades de la última cifra. Se recorren cuatro cifras y los cuatro operadores; después se despeja algebraicamente la quinta cifra necesaria.

La evaluación se mantiene como `suma_completada + termino_actual`, lo que respeta la precedencia de `*` y `/`. Por ejemplo, si el último operador es `*`, la igualdad es

$$T=\text{suma\_completada}+\text{termino\_actual}\cdot e,$$

y se despeja $e$. Se utiliza `Fraction` para evitar errores de coma flotante. Finalmente se comprueba que $e$ sea una cifra entera del 1 al 9, que no esté repetida y que la expresión realmente valga $T$.

### Pseudocódigo del algoritmo mejorado

ALGORITMO BuscarObjetivoMejorado(cifras, operadores, objetivo)
    prefijos_probados ← 0

    PARA CADA primeros_cuatro EN Permutaciones(cifras, 4) HACER
        PARA CADA operaciones EN Permutaciones(operadores, 4) HACER
            prefijos_probados ← prefijos_probados + 1

            // Evaluar las cuatro primeras cifras respetando la precedencia
            suma ← 0
            termino ← primeros_cuatro[0]
            PARA i ← 1 HASTA 3 HACER
                (suma, termino) ← ActualizarEstado(
                    suma, termino, operaciones[i - 1], primeros_cuatro[i]
                )
            FIN PARA

            ultimo_operador ← operaciones[3]
            valor_prefijo ← suma + termino

            // Despejar la única quinta cifra que podría alcanzar el objetivo
            SEGÚN ultimo_operador HACER
                CASO "+": quinta ← objetivo - valor_prefijo
                CASO "-": quinta ← valor_prefijo - objetivo
                CASO "*": quinta ← (objetivo - suma) / termino
                CASO "/":
                    SI objetivo = suma ENTONCES CONTINUAR
                    quinta ← termino / (objetivo - suma)
            FIN SEGÚN

            SI quinta es una cifra entera disponible
               Y quinta NO ESTÁ EN primeros_cuatro ENTONCES
                numeros ← primeros_cuatro seguidos de quinta
                expresion ← ConstruirExpresion(numeros, operaciones)

                SI EvaluarExactamente(expresion) = objetivo ENTONCES
                    DEVOLVER expresion, prefijos_probados
                FIN SI
            FIN SI
        FIN PARA
    FIN PARA

    DEVOLVER NULO, prefijos_probados
FIN ALGORITMO

A diferencia de la fuerza bruta, este algoritmo no recorre la quinta cifra: la obtiene mediante una ecuación y descarta el prefijo si el valor calculado no es una cifra válida y no repetida.

In [6]:
from fractions import Fraction


def actualizar_estado(suma, termino, operador, numero):
    """Procesa un operador conservando la precedencia aritmética."""
    numero = Fraction(numero)
    if operador == "+":
        return suma + termino, numero
    if operador == "-":
        return suma + termino, -numero
    if operador == "*":
        return suma, termino * numero
    return suma, termino / numero


def evaluar_exactamente(numeros, operaciones):
    """Evalúa la expresión sin redondeos y con precedencia habitual."""
    suma, termino = Fraction(0), Fraction(numeros[0])
    for operador, numero in zip(operaciones, numeros[1:]):
        suma, termino = actualizar_estado(suma, termino, operador, numero)
    return suma + termino


def buscar_objetivo_mejorado(objetivo, cifras_disponibles=range(1, 10)):
    """Busca una solución despejando la quinta cifra."""
    objetivo = Fraction(objetivo)
    cifras_disponibles = tuple(cifras_disponibles)
    conjunto_cifras = set(cifras_disponibles)
    prefijos_probados = 0

    for primeros_cuatro in itertools.permutations(cifras_disponibles, 4):
        usados = set(primeros_cuatro)
        for operaciones in itertools.permutations(operadores):
            prefijos_probados += 1
            suma, termino = Fraction(0), Fraction(primeros_cuatro[0])
            for operador, numero in zip(operaciones[:3], primeros_cuatro[1:]):
                suma, termino = actualizar_estado(suma, termino, operador, numero)

            ultimo_operador = operaciones[3]
            valor_prefijo = suma + termino

            if ultimo_operador == "+":
                quinta = objetivo - valor_prefijo
            elif ultimo_operador == "-":
                quinta = valor_prefijo - objetivo
            elif ultimo_operador == "*":
                quinta = (objetivo - suma) / termino
            else:  # objetivo = suma + termino / quinta
                if objetivo == suma:
                    continue
                quinta = termino / (objetivo - suma)

            if quinta.denominator != 1:
                continue
            quinta = int(quinta)
            if quinta not in conjunto_cifras or quinta in usados:
                continue

            numeros = primeros_cuatro + (quinta,)
            if evaluar_exactamente(numeros, operaciones) == objetivo:
                return construir_expresion(numeros, operaciones), prefijos_probados

    return None, prefijos_probados


solucion_mejorada, prefijos = buscar_objetivo_mejorado(objetivo)
print(f"Solución: {solucion_mejorada} = {objetivo}")
print("Prefijos examinados hasta encontrarla:", prefijos)

Solución: 1+2*6-9/3 = 10
Prefijos examinados hasta encontrarla: 555


### 11. (*) Calcule la complejidad del algoritmo.

En el peor caso se examinan

$$P(n,4)\cdot4!$$

prefijos. Cada despeje y validación cuesta tiempo constante, por lo que la complejidad es **$\Theta(n^4)$** y el espacio auxiliar es **$O(1)$** si se devuelve la primera solución. Para las nueve cifras son, como máximo, $P(9,4)\cdot24=72\,576$ prefijos: una quinta parte de las 362.880 expresiones de la fuerza bruta.

La mejora es completa para la búsqueda de un objetivo: no descarta ninguna solución válida, porque para cada prefijo y último operador calcula la única quinta cifra que podría completar la igualdad. Para enumerar *todos* los valores y sus extremos sí se conserva la fuerza bruta, pues esa tarea requiere explorar el espacio global y no un único objetivo.

In [7]:
import math 
maximo_prefijos = math.perm(9, 4) * math.factorial(4)
print("Máximo de prefijos del algoritmo mejorado:", maximo_prefijos)
print("Factor de reducción frente a fuerza bruta:", total_expresiones / maximo_prefijos)

Máximo de prefijos del algoritmo mejorado: 72576
Factor de reducción frente a fuerza bruta: 5.0


### Comparación cuantitativa de los dos algoritmos

| Criterio | Fuerza bruta | Algoritmo mejorado | Mejora obtenida |
|---|---:|---:|---:|
| Complejidad temporal | $\Theta(n^5)$ | $\Theta(n^4)$ | Se elimina un factor lineal $n$ |
| Candidatos en el peor caso ($n=9$) | 362.880 expresiones | 72.576 prefijos | 290.304 candidatos menos (80 %) |
| Relación entre máximos | 5 veces más candidatos | 1 vez | Reducción por un factor de 5 |
| Memoria auxiliar para hallar una solución | $O(1)$ | $O(1)$ | No aumenta |
| Garantía de encontrar una solución | Sí | Sí | Se conserva la completitud |

La reducción no procede de utilizar un equipo más rápido, sino de evitar trabajo: la fuerza bruta prueba cada posible quinta cifra, mientras que el algoritmo mejorado **calcula directamente la única quinta cifra que podría funcionar**. Con nueve cifras, el máximo teórico disminuye un 80 %. Al crecer $n$, la relación entre ambos espacios es

$$\frac{P(n,5)}{P(n,4)}=n-4,$$

por lo que la ventaja también crece con el tamaño de entrada. La siguiente prueba mide ambos algoritmos bajo las mismas condiciones: mismo objetivo, detención en la primera solución y mediana de siete ejecuciones. El tiempo exacto depende del ordenador, pero el número de candidatos es independiente del equipo.

### 12. Según el problema y cuando tenga sentido, diseñe un juego de datos de entrada aleatorios.

Las cifras y operadores forman parte fija del enunciado; modificarlos produciría otro problema. La entrada que sí cambia es el entero objetivo. Se genera de manera reproducible dentro del intervalo alcanzable usando una semilla.

In [8]:
import random
generador = random.Random(42)
objetivo_aleatorio = generador.randint(valor_minimo, valor_maximo)
objetivo_aleatorio

-41

### 13. Aplique el algoritmo al juego de datos generado.

In [9]:
solucion_aleatoria, prefijos_aleatorios = buscar_objetivo_mejorado(objetivo_aleatorio)
print("Objetivo aleatorio:", objetivo_aleatorio)
print("Solución:", solucion_aleatoria)
print("Comprobación exacta con eval:", eval(solucion_aleatoria))
print("Prefijos examinados:", prefijos_aleatorios)

Objetivo aleatorio: -41
Solución: 1-5*9+6/2
Comprobación exacta con eval: -41.0
Prefijos examinados: 3969


Con la semilla elegida se obtiene el objetivo **−41** y una solución es `1-5*9+6/2 = -41`. Como se demostró antes que no faltan enteros en $[-69,77]$, cualquier objetivo generado dentro de ese intervalo tendrá al menos una solución.

## Referencias

- Universidad Internacional de Valencia. (s. f.). VC4 – Problemas del trabajo práctico, 03MIAR – Algoritmos de optimización (Problema 3 y preguntas del entregable).

- Python Software Foundation. (s. f.). itertools.permutations. Python documentation. https://docs.python.org/3/library/itertools.html#itertools.permutations

- Python Software Foundation. (s. f.). fractions.Fraction. Python documentation. https://docs.python.org/3/library/fractions.html

- Python Software Foundation. (s. f.). eval. Python documentation. https://docs.python.org/3/library/functions.html#eval


Para el desarrollo de la actividad utilicé herramientas de inteligencia artificial generativa, como ChatGPT y Claude, como apoyo para revisar y comprender conceptos, explorar ideas o nuevas formas de abordar los problemas y estructurar los contenidos de una manera diferente. Sin embargo, fui crítico con todas las respuestas obtenidas y construí mi propio razonamiento para el desarrollo de la actividad.

## Describa brevemente cómo es posible avanzar en el estudio del problema. Considere variaciones del problema y aumentos de tamaño.

El estudio puede ampliarse aumentando el conjunto de cifras, permitiendo más operadores, incorporando el cero o admitiendo paréntesis. Esta última variante cambia notablemente el espacio de búsqueda porque también deben considerarse las distintas agrupaciones. Para tamaños mayores convendría usar programación dinámica por subconjuntos: almacenar resultados exactos alcanzables por cada conjunto de cifras y operadores evita recalcular subexpresiones equivalentes. También sería útil comparar tiempos y consumo de memoria de fuerza bruta, despeje de la última cifra y programación dinámica para distintos tamaños de entrada.

## Conclusión



Se eligió **despejar la quinta cifra** en lugar de aplicar programación dinámica porque este problema tiene un dominio finito, bien definido y relativamente pequeño: nueve cifras, cinco posiciones y cuatro operadores. En estas condiciones, el despeje es una solución más sencilla de implementar y explicar, utiliza memoria auxiliar $O(1)$ y produce una mejora medible: reduce la complejidad de la búsqueda de un objetivo de $\Theta(n^5)$ a $\Theta(n^4)$ y, para la instancia del enunciado, disminuye el máximo de candidatos de 362.880 a 72.576, es decir, un 80 %.

La programación dinámica necesitaría almacenar estados y resultados parciales, por lo que añade consumo de memoria y complejidad de implementación que no se justifican para esta instancia. Sin embargo, no debe afirmarse que sea siempre «más óptima» para cualquier dominio grande. Puede resultar más adecuada cuando el número de cifras u operadores crece, se permiten paréntesis y distintas agrupaciones, se consultan muchos objetivos sobre el mismo dominio o aparecen numerosos subproblemas repetidos. En esos casos, reutilizar resultados almacenados puede compensar ampliamente el coste adicional de memoria. Por tanto, el despeje es la alternativa más proporcionada para el problema actual, mientras que la programación dinámica constituye una posible estrategia de escalado para variantes de mayor tamaño o estructura más compleja.